# 証券営業インテリジェンス ハンズオン
## Part 1: ダミーデータ生成

このノートブックでは、ハンズオン用のダミーデータを生成します。
JavaScript ストアドプロシージャを使用して、50人分の顧客データ・ポートフォリオ・取引履歴などをランダムに生成します。

> **Note**: このデータはすべて架空のものです。実際の顧客情報は一切含まれていません。

In [ ]:
%%sql -r result_use
USE DATABASE SNOWFINANCE_DB;
USE SCHEMA DEMO_SCHEMA;
USE WAREHOUSE DEMO_WH;
SELECT 'Ready!' AS STATUS;

## Step 1: 顧客マスタデータ投入

まず、山田太郎様（デモの主役）と少数のサンプルデータを手動で投入します。
その後、ストアドプロシージャで残りの顧客データを自動生成します。

In [ ]:
%%sql -r result_c001
-- ============================================================
-- デモシナリオの主役: C001 山田太郎様（68歳, 元上場企業役員）
-- ============================================================
INSERT INTO DIM_CUSTOMER VALUES (
    'C001', '山田太郎', 'ヤマダタロウ', 68, '男性', '1957-04-15',
    '元会社役員', '元トヨタ関連会社', '元代表取締役', '東京都', '千代田区',
    '3000万円以上', 800000000, 200000000, '保守的', '資産保全',
    30, 'プライベートバンク', 'RM001', '鈴木一郎',
    '1995-04-01', '2025-01-10',
    FALSE, FALSE, FALSE,
    '孫の海外留学費用2,000万円が必要。トヨタ株を多数保有。相続対策にも関心あり。',
    CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP()
);

-- 山田様の家族情報
INSERT INTO DIM_FAMILY VALUES
('F001', 'C001', '配偶者', '山田花子', 65, '主婦', TRUE, 1, NULL, CURRENT_TIMESTAMP()),
('F002', 'C001', '長男',   '山田次郎', 40, '会社員', TRUE, 2, NULL, CURRENT_TIMESTAMP()),
('F003', 'C001', '長女',   '田中明子', 38, '医師',   TRUE, 3, NULL, CURRENT_TIMESTAMP()),
('F004', 'C001', '孫（長男の子）', '山田健太', 17, '高校生', FALSE, NULL, '海外留学予定', CURRENT_TIMESTAMP());

-- 山田様のライフイベント
INSERT INTO DIM_LIFE_EVENT VALUES
('E001', 'C001', '教育資金', '孫（山田健太）の米国大学留学費用。4年間で約2,000万円が必要。2025年9月入学予定。', '2025-09-01', 20000000, '高', '検討中', 'F004', '2025-01-10', '2025-01-10', 'トヨタ株売却か、ローン利用か検討中'),
('E002', 'C001', '相続対策', '相続税対策の見直し。現在の資産構成では相続税が高額になる見込み。', '2025-12-31', NULL, '中', '検討中', NULL, '2024-06-01', '2025-01-10', '生前贈与の活用を検討');

SELECT '山田太郎様データ投入完了!' AS STATUS;

In [ ]:
%%sql -r result_portfolio_c001
-- 山田様のポートフォリオ（トヨタ株を大量保有）
INSERT INTO FACT_PORTFOLIO VALUES
('P0001', 'C001', '国内株式', '7203', 'トヨタ自動車', 10000, 1500.00, '2010-04-01', 2850.00, 28500000, 13500000, 90.00, '特定口座', 'JPY', '2025-01-15'),
('P0002', 'C001', '国内株式', '6758', 'ソニーグループ', 5000, 8000.00, '2015-06-01', 15200.00, 76000000, 36000000, 90.00, '特定口座', 'JPY', '2025-01-15'),
('P0003', 'C001', '国内株式', '9433', 'KDDI', 8000, 2800.00, '2012-04-01', 4650.00, 37200000, 14800000, 62.86, '特定口座', 'JPY', '2025-01-15'),
('P0004', 'C001', '国内株式', '8058', '三菱商事', 6000, 2000.00, '2018-04-01', 3200.00, 19200000, 7200000, 60.00, '特定口座', 'JPY', '2025-01-15'),
('P0005', 'C001', '海外株式', 'AAPL', 'Apple Inc.', 500, 120.00, '2019-01-01', 185.00, 13875000, 4875000, 54.17, '特定口座', 'JPY', '2025-01-15'),
('P0006', 'C001', '投資信託', 'INF001', '日経225インデックスファンド', 100000, 18000.00, '2014-07-01', 25000.00, 25000000, 7000000, 38.89, '積立NISA', 'JPY', '2025-01-15'),
('P0007', 'C001', '国内債券', 'BD001', '個人向け国債（変動10年）', 50000000, 1.00, '2020-04-01', 1.00, 50000000, 0, 0.00, '一般口座', 'JPY', '2025-01-15'),
('P0008', 'C001', '現金・預金', 'CASH', '普通預金・定期預金', 1, 100000000, '2025-01-01', 100000000, 100000000, 0, 0.00, '一般口座', 'JPY', '2025-01-15');

SELECT '山田様ポートフォリオ投入完了!' AS STATUS, SUM(MARKET_VALUE) AS 時価総額
FROM FACT_PORTFOLIO WHERE CUSTOMER_ID = 'C001';

## Step 2: ストアドプロシージャで顧客データを一括生成

JavaScript ストアドプロシージャを使用して、残り49人分の顧客データを自動生成します。

> **処理内容**: 職業・資産・リスク許容度・RM割り当てなどをランダムに生成

In [ ]:
%%sql -r result_gen_customers
-- 顧客一括生成ストアドプロシージャ
CREATE OR REPLACE PROCEDURE GENERATE_CUSTOMER_DATA()
RETURNS VARCHAR
LANGUAGE JAVASCRIPT
AS
$$
var segments = ['プライベートバンク', 'ゴールド', 'シルバー', 'レギュラー'];
var occupations = ['経営者', '医師', '弁護士', '公認会計士', '不動産オーナー', '資産家', '元会社役員', '大学教授'];
var prefectures = ['東京都', '神奈川県', '大阪府', '愛知県', '埼玉県', '千葉県', '兵庫県', '福岡県'];
var purposes = ['資産保全', '資産形成', '相続対策', '教育資金', '老後資金'];
var risks = ['保守的', 'やや保守的', 'バランス型', 'やや積極的', '積極的'];
var rms = ['RM001', 'RM002', 'RM003', 'RM004', 'RM005', 'RM006'];
var rmNames = {'RM001':'鈴木一郎','RM002':'田中二郎','RM003':'佐藤三郎','RM004':'高橋四郎','RM005':'伊藤五郎','RM006':'渡辺六郎'};

var count = 0;
for (var i = 2; i <= 50; i++) {
    var cid = 'C' + String(i).padStart(3, '0');
    var seg = segments[Math.floor(Math.random() * 4)];
    var assets = seg === 'プライベートバンク' ? Math.floor(Math.random() * 9000000000) + 1000000000
               : seg === 'ゴールド' ? Math.floor(Math.random() * 900000000) + 100000000
               : seg === 'シルバー' ? Math.floor(Math.random() * 90000000) + 10000000
               : Math.floor(Math.random() * 9000000) + 1000000;
    var occ = occupations[Math.floor(Math.random() * occupations.length)];
    var pref = prefectures[Math.floor(Math.random() * prefectures.length)];
    var purpose = purposes[Math.floor(Math.random() * purposes.length)];
    var risk = risks[Math.floor(Math.random() * risks.length)];
    var rm = rms[Math.floor(Math.random() * rms.length)];
    var age = Math.floor(Math.random() * 40) + 45;
    var sql = `INSERT INTO DIM_CUSTOMER (CUSTOMER_ID, CUSTOMER_NAME, AGE, GENDER, OCCUPATION, PREFECTURE, TOTAL_ASSETS, LIQUID_ASSETS, RISK_TOLERANCE, INVESTMENT_PURPOSE, INVESTMENT_EXPERIENCE_YEARS, SEGMENT, RM_ID, RM_NAME, ACCOUNT_OPEN_DATE, LAST_CONTACT_DATE, CREATED_AT, UPDATED_AT)
    VALUES ('${cid}', '顧客${i}号', ${age}, '${Math.random()>0.3?"男性":"女性"}', '${occ}', '${pref}', ${assets}, ${Math.floor(assets*0.3)}, '${risk}', '${purpose}', ${Math.floor(Math.random()*30)+5}, '${seg}', '${rm}', '${rmNames[rm]}', '${2010+Math.floor(Math.random()*10)}-${String(Math.floor(Math.random()*12)+1).padStart(2,'0')}-01', '2025-01-15', CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP())`;
    try { snowflake.execute({sqlText: sql}); count++; } catch(e) {}
}
return 'Generated ' + count + ' customers';
$$;

CALL GENERATE_CUSTOMER_DATA();
SELECT COUNT(*) AS 顧客数 FROM DIM_CUSTOMER;

In [ ]:
%%sql -r result_gen_portfolio
-- ポートフォリオデータ生成ストアドプロシージャ
CREATE OR REPLACE PROCEDURE GENERATE_PORTFOLIO_DATA()
RETURNS VARCHAR
LANGUAGE JAVASCRIPT
AS
$$
var securities = [
    {class:'国内株式', code:'7203', name:'トヨタ自動車', price:2850},
    {class:'国内株式', code:'6758', name:'ソニーグループ', price:15200},
    {class:'国内株式', code:'9433', name:'KDDI', price:4650},
    {class:'国内株式', code:'8058', name:'三菱商事', price:3200},
    {class:'海外株式', code:'AAPL', name:'Apple Inc.', price:185},
    {class:'海外株式', code:'MSFT', name:'Microsoft Corp.', price:420},
    {class:'投資信託', code:'INF001', name:'日経225インデックスファンド', price:25000},
    {class:'投資信託', code:'INF002', name:'グローバル株式インデックスファンド', price:18000}
];
var count = 0; var pid = 9; // P0001-P0008 は C001 で使用済み
for (var i = 2; i <= 50; i++) {
    var cid = 'C' + String(i).padStart(3, '0');
    var numHoldings = Math.floor(Math.random() * 5) + 1;
    for (var j = 0; j < numHoldings; j++) {
        var sec = securities[Math.floor(Math.random() * securities.length)];
        var qty = Math.floor(Math.random() * 10000) + 100;
        var acqPrice = sec.price * (0.7 + Math.random() * 0.6);
        var mv = Math.floor(qty * sec.price);
        var gain = Math.floor(mv - qty * acqPrice);
        var gainPct = Math.round(gain / (qty * acqPrice) * 10000) / 100;
        var portfolioId = 'P' + String(pid).padStart(4, '0');
        var sql = `INSERT INTO FACT_PORTFOLIO VALUES ('${portfolioId}','${cid}','${sec.class}','${sec.code}','${sec.name}',${qty},${Math.round(acqPrice*100)/100},'2020-01-01',${sec.price},${mv},${gain},${gainPct},'特定口座','JPY','2025-01-15')`;
        try { snowflake.execute({sqlText: sql}); count++; pid++; } catch(e) {}
    }
}
return 'Generated ' + count + ' portfolio records';
$$;

CALL GENERATE_PORTFOLIO_DATA();
SELECT COUNT(*) AS ポートフォリオ件数 FROM FACT_PORTFOLIO;

In [ ]:
%%sql -r result_gen_transactions
-- 取引履歴データ生成
CREATE OR REPLACE PROCEDURE GENERATE_TRANSACTION_DATA()
RETURNS VARCHAR
LANGUAGE JAVASCRIPT
AS
$$
var txnTypes = ['買付', '売却', '配当', '利金', '買付'];
var channels = ['店頭', '電話', 'Web', 'アプリ', '店頭'];
var securities = [
    {class:'国内株式', code:'7203', name:'トヨタ自動車'},
    {class:'国内株式', code:'6758', name:'ソニーグループ'},
    {class:'国内株式', code:'9433', name:'KDDI'},
    {class:'国内株式', code:'8058', name:'三菱商事'},
    {class:'海外株式', code:'AAPL', name:'Apple Inc.'},
    {class:'投資信託', code:'INF001', name:'日経225インデックスファンド'},
    {class:'投資信託', code:'INF002', name:'グローバル株式インデックスファンド'}
];
var count = 0; var txnId = 11;
// 山田様の取引履歴（T001-T010）を先に手動データ（別途実行）として残し、T011以降を生成
for (var i = 0; i < 150; i++) {
    var tid = 'T' + String(txnId).padStart(4, '0');
    var custNum = Math.floor(Math.random() * 50) + 1;
    var cid = 'C' + String(custNum).padStart(3, '0');
    var year = Math.random() > 0.1 ? 2024 : 2025;
    var month = year === 2025 ? 1 : Math.floor(Math.random()*12)+1;
    var day = Math.floor(Math.random()*28)+1;
    var txnDate = year + '-' + String(month).padStart(2,'0') + '-' + String(day).padStart(2,'0');
    var txnType = txnTypes[Math.floor(Math.random() * txnTypes.length)];
    var sec = securities[Math.floor(Math.random() * securities.length)];
    var qty = Math.floor(Math.random()*5000)+100;
    var price = Math.floor(Math.random()*50000)+1000;
    var amount = Math.min(qty*price, 100000000);
    var fee = Math.floor(amount*0.001);
    var tax = txnType==='売却' ? Math.floor(amount*0.02) : 0;
    var net = txnType==='売却' ? amount-fee-tax : amount+fee;
    var rmId = 'RM' + String(Math.floor(Math.random()*6)+1).padStart(3,'0');
    var sql = `INSERT INTO FACT_TRANSACTION (TRANSACTION_ID,CUSTOMER_ID,TRANSACTION_DATE,TRANSACTION_TYPE,ASSET_CLASS,SECURITY_CODE,SECURITY_NAME,QUANTITY,PRICE,AMOUNT,FEE,TAX,NET_AMOUNT,ACCOUNT_TYPE,ORDER_CHANNEL,RM_ID)
    VALUES ('${tid}','${cid}','${txnDate}','${txnType}','${sec.class}','${sec.code}','${sec.name}',${qty},${price},${amount},${fee},${tax},${net},'特定口座','${channels[Math.floor(Math.random()*channels.length)]}','${rmId}')`;
    try { snowflake.execute({sqlText: sql}); count++; txnId++; } catch(e) {}
}
return 'Generated ' + count + ' transactions';
$$;

-- 山田様の取引履歴（10件）
INSERT INTO FACT_TRANSACTION VALUES
('T0001','C001','2024-12-01','買付','国内株式','7203','トヨタ自動車',500,2780,1390000,1390,0,1391390,'特定口座','店頭','RM001'),
('T0002','C001','2024-11-15','売却','国内株式','6758','ソニーグループ',200,14800,2960000,2960,592000,2365040,'特定口座','電話','RM001'),
('T0003','C001','2024-10-01','配当','国内株式','7203','トヨタ自動車',10000,40,400000,0,80000,320000,'特定口座','自動','RM001'),
('T0004','C001','2024-09-01','買付','投資信託','INF001','日経225インデックスファンド',1000,23000,23000000,23000,0,23023000,'積立NISA','Web','RM001'),
('T0005','C001','2024-08-01','買付','国内株式','9433','KDDI',1000,4400,4400000,4400,0,4404400,'特定口座','電話','RM001'),
('T0006','C001','2024-07-15','配当','国内株式','9433','KDDI',8000,72,576000,0,115200,460800,'特定口座','自動','RM001'),
('T0007','C001','2024-06-01','買付','海外株式','AAPL','Apple Inc.',100,170,1700000,1700,0,1701700,'特定口座','Web','RM001'),
('T0008','C001','2024-05-01','配当','国内株式','8058','三菱商事',6000,100,600000,0,120000,480000,'特定口座','自動','RM001'),
('T0009','C001','2024-04-01','買付','国内債券','BD001','個人向け国債',50000000,1,50000000,0,0,50000000,'一般口座','店頭','RM001'),
('T0010','C001','2024-03-01','売却','国内株式','7203','トヨタ自動車',300,2750,825000,825,165000,659175,'特定口座','電話','RM001');

CALL GENERATE_TRANSACTION_DATA();
SELECT COUNT(*) AS 取引件数 FROM FACT_TRANSACTION;

## Step 3: 信託銀行商品・ニュース・アナリストレポートの投入

これらのデータは `setup.sql` の Part 4〜5 に含まれています。
以下のセルで `setup.sql` の該当部分を実行してください。

> **Tip**: 大量のINSERT文を実行するため、`setup.sql` ファイルをSnowsightのワークシートで実行することを推奨します。

In [ ]:
%%sql -r result_verify_all
-- データ確認
SELECT TABLE_NAME, ROW_COUNT
FROM INFORMATION_SCHEMA.TABLES
WHERE TABLE_SCHEMA = 'DEMO_SCHEMA' AND TABLE_TYPE = 'BASE TABLE'
ORDER BY TABLE_NAME;

## まとめ

Part 1 完了！ダミーデータの生成が完了しました。

| テーブル | 件数 |
|---|---|
| DIM_CUSTOMER | 50人（C001含む） |
| DIM_FAMILY | 約100件 |
| DIM_LIFE_EVENT | 約50件 |
| FACT_PORTFOLIO | 約250件 |
| FACT_TRANSACTION | 約160件 |

**次のステップ:** `02_ai_functions.ipynb` でCortex AI Functionsを体験してください。